# MapClass / GeoViLM — GPU Training Playground

Build & train a checkpoint from scratch on **GPU** so the inference playground (`playground.ipynb`) has something to load. Two backbones in one notebook:

1. **Mock backbone** — tiny conv stub from `mapclass.model.mock_backbone`. Writes a checkpoint to `models/geovilm_phase1_mock.pt` that `playground.ipynb` can pick up directly. This is the Phase 1 path the project's `train.py` already supports.
2. **PaliGemma backbone** — defined inline here (Phase 5 / MODEL-04 isn't merged yet; `train.py` + `infer.load_model` both refuse non-mock backbones). The PaliGemma cells construct the backbone, train heads on top of a frozen vision tower, and run inference in-process so we bypass the public guards.

Heads up:
- PaliGemma is **gated on HuggingFace** — run `huggingface-cli login` and accept the license at https://huggingface.co/google/paligemma-3b-mix-224 before the PaliGemma cells.
- The Mock checkpoint produced here matches the schema `playground.ipynb` expects (taxonomy hash, processor identity, etc.). The PaliGemma checkpoint does **not** — `mapclass.infer.load_model` will refuse it. The PaliGemma section runs inference inline.

## 1. Setup — GPU detection, imports, sys.path

In [ ]:
import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path

REPO = Path('/home/drdreadknee/mapclass')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
import numpy as np
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch: {torch.__version__}')
print(f'device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  name:     {torch.cuda.get_device_name(0)}')
    print(f'  vram_gb:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}')
    print(f'  capability: {torch.cuda.get_device_capability(0)}')

## 2. Make sure fixture data exists

The trainer reads samples from `data/renders/`. If empty, run the smoke-fixture generator (synthetic 64×64 PNGs whose RGB encodes labels — the Mock backbone has a learnable signal).

In [ ]:
RENDERS = REPO / 'data' / 'renders'
needs_gen = (not RENDERS.exists()) or (not any(RENDERS.iterdir()))
if needs_gen:
    print('Generating fixtures via scripts/_make_smoke_fixtures.py ...')
    out = subprocess.run(
        [sys.executable, '-m', 'scripts._make_smoke_fixtures'],
        cwd=REPO, capture_output=True, text=True,
    )
    print(out.stdout[-2000:] if out.stdout else '(no stdout)')
    if out.returncode != 0:
        print('STDERR:', out.stderr[-2000:])
        raise RuntimeError(f'fixture generation failed (rc={out.returncode})')
n_samples = sum(1 for p in RENDERS.iterdir() if p.is_dir())
print(f'data/renders/ has {n_samples} samples')

## 3. Build the Mock model & train on GPU

Uses the project's existing `MockBackbone`, `GeoViLM`, and seg heads — same architecture the inference playground expects. Training loop is inlined (instead of calling `python -m mapclass.train`) so you can step through batches and tweak hyperparams.

In [ ]:
from torch import nn
from torch.utils.data import DataLoader
from safetensors.torch import save_file

from mapclass.data.dataset import MapClassDataset
from mapclass.data.loss_weights import LossWeights
from mapclass.data.splits import load_splits, build_splits, write_splits
from mapclass.data.taxonomy import taxonomy_hash
from mapclass.model.mock_backbone import MockBackbone
from mapclass.model.geovilm import GeoViLM
from mapclass.model.seg_heads import LandCoverHead, TopographyHead
from mapclass.seeding import set_global_seed

SEED = 42
N_EPOCHS = 3        # bump for less-noisy weights
BATCH_SIZE = 8
LR = 1e-3

set_global_seed(SEED)

# Splits — load if committed, else build + commit them.
splits_path = REPO / 'mapclass/configs/splits.json'
sample_ids = sorted(p.name for p in RENDERS.iterdir() if p.is_dir())
if splits_path.is_file():
    splits = load_splits(splits_path)
else:
    splits = build_splits(sample_ids)
    write_splits(splits, splits_path)
train_ids = splits['by_split']['train']
print(f'train split: {len(train_ids)} samples')

train_ds = MapClassDataset(train_ids, root=RENDERS, validate=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Build model on GPU.
mock_bb = MockBackbone(training_seed=SEED)
n_feat = mock_bb.feature_channels['features']
model = GeoViLM(mock_bb, LandCoverHead(in_channels=n_feat), TopographyHead(in_channels=n_feat)).to(DEVICE)
optim = torch.optim.Adam(model.parameters(), lr=LR)
print(f'model params: {sum(p.numel() for p in model.parameters()):,}')
print(f'on device:    {next(model.parameters()).device}')

In [ ]:
def compute_loss(out, batch):
    lc_target = batch['land_cover'].to(DEVICE)
    topo_target = batch['topography'].to(DEVICE)
    lc_logits = out['land_cover_logits']
    topo_logits = out['topography_logits']
    # Some backbones (PaliGemma) resize inputs to 224×224 in preprocess(), so
    # GeoViLM's upsample-to-input-size yields logits at 224×224 while labels
    # remain at the fixture resolution (e.g. 64×64). Downsample logits to the
    # label resolution before CE. No-op for the Mock path (sizes already match).
    if lc_logits.shape[-2:] != lc_target.shape[-2:]:
        lc_logits = nn.functional.interpolate(lc_logits, size=lc_target.shape[-2:], mode='bilinear', align_corners=False)
        topo_logits = nn.functional.interpolate(topo_logits, size=topo_target.shape[-2:], mode='bilinear', align_corners=False)
    ce_lc = nn.functional.cross_entropy(lc_logits, lc_target, ignore_index=255)
    ce_topo = nn.functional.cross_entropy(topo_logits, topo_target, ignore_index=255)
    return ce_lc + ce_topo

model.train()
for epoch in range(1, N_EPOCHS + 1):
    epoch_losses = []
    for step, batch in enumerate(train_loader):
        image = model.backbone.preprocess(batch['image'].to(DEVICE))
        out = model(image)
        loss = compute_loss(out, batch)
        optim.zero_grad()
        loss.backward()
        optim.step()
        epoch_losses.append(loss.item())
    print(f'epoch {epoch}/{N_EPOCHS}: mean_loss={sum(epoch_losses)/len(epoch_losses):.4f}  (first={epoch_losses[0]:.4f}  last={epoch_losses[-1]:.4f})')


## 4. Save checkpoint (compatible with `playground.ipynb`)

Writes the same safetensors metadata schema the inference playground checks — `playground.ipynb` will load this directly via `mapclass.infer.load_model`.

In [ ]:
lw = LossWeights.load(REPO / 'mapclass/configs/loss_weights.yaml')
dataset_manifest_sha = hashlib.sha256('\n'.join(sorted(sample_ids)).encode()).hexdigest()

raw_meta = {
    'model_version':             '0.1.0',
    'dataset_manifest_sha':      dataset_manifest_sha,
    'taxonomy_hash':             taxonomy_hash(),
    'training_seed':             SEED,
    'source_class_weights_hash': lw.source_class_weights_hash,
    'backbone':                  'mock',
    'processor_identity':        model.backbone.processor_identity,
}
metadata = {k: json.dumps(v) for k, v in raw_meta.items()}
out_path = REPO / 'models/geovilm_phase1_mock.pt'
out_path.parent.mkdir(parents=True, exist_ok=True)
# safetensors.save_file requires CPU tensors
state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
save_file(state, str(out_path), metadata=metadata)
print(f'wrote: {out_path}  ({out_path.stat().st_size/1e6:.2f} MB)')

## 5. Sanity-check via the public load_model API

Round-trips through `mapclass.infer.load_model` — the same path `playground.ipynb` uses. If this cell prints a prediction, `playground.ipynb` will work.

In [ ]:
from mapclass.infer import load_model

predictor = load_model(out_path, device='cuda' if DEVICE.type == 'cuda' else 'cpu')
sample_dir = RENDERS / sorted(p.name for p in RENDERS.iterdir() if p.is_dir())[0]
img_path = next(p for p in sample_dir.iterdir() if p.name in ('image.png', 'flat.png', 'illustrated.png', 'satellite.png'))
img = Image.open(img_path).convert('RGB')
result = predictor.predict(img)
print('result keys:', list(result.keys()))
print(f"  land_cover shape:       {tuple(result['land_cover'].shape)}")
print(f"  topography shape:       {tuple(result['topography'].shape)}")
print(f"  land_cover unique args: {result['land_cover_argmax'].unique().tolist()}")
print(f"  model_version:          {result['model_version']}")

---

# PaliGemma section

Everything from here on is a Phase 5 / MODEL-04 *preview* — `mapclass/model/` doesn't ship a `PaliGemma2Backbone` yet, and `train.py` / `infer.load_model` will both refuse a non-mock checkpoint. We build the backbone inline, train heads on a frozen vision tower, and do inference in-process.

## 6. Deps, auth, and GCS-cached weights

PaliGemma is a **gated** model on HuggingFace. First-time setup (done already on this VM):
1. Open https://huggingface.co/google/paligemma-3b-mix-224 and accept the license.
2. `.venv/bin/hf auth login --token <HF_TOKEN>` (token with read scope).
3. `uv sync --extra notebook --extra paligemma` to install `sentencepiece`.

**Weights are cached in GCS** at `gs://mapclass-training-northeast1/models/paligemma-3b-mix-224/` (10.91 GiB across 13 files). The cell below rsyncs them to `/tmp/paligemma-3b-mix-224/` if not already present. On this VM the rsync is a no-op; on a fresh VM it's a ~1-2 min within-region (or cross-region) GCP transfer. After the sync, `from_pretrained()` is given a local path — no HF auth required at load time.


In [ ]:
from huggingface_hub import HfFolder
import importlib

missing = []
for pkg in ('sentencepiece', 'transformers', 'accelerate', 'peft', 'bitsandbytes'):
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)
if missing:
    raise RuntimeError(f'missing packages: {missing}. Run: uv sync --extra notebook --extra paligemma')

# Sync PaliGemma weights from GCS to local tmp (idempotent). Avoids re-downloading
# from HF on every fresh VM and removes the need for HF auth at model-load time.
PALIGEMMA_GCS_URI = 'gs://mapclass-training-northeast1/models/paligemma-3b-mix-224'
PALIGEMMA_PATH = Path('/tmp/paligemma-3b-mix-224')

required_files = ['config.json', 'tokenizer.json', 'preprocessor_config.json',
                  'model-00001-of-00003.safetensors',
                  'model-00002-of-00003.safetensors',
                  'model-00003-of-00003.safetensors']
have_all = PALIGEMMA_PATH.exists() and all((PALIGEMMA_PATH / f).exists() for f in required_files)

if have_all:
    print(f'PaliGemma weights present at {PALIGEMMA_PATH} — skipping sync.')
else:
    print(f'Syncing {PALIGEMMA_GCS_URI} -> {PALIGEMMA_PATH} (one-time, ~11 GB) ...')
    PALIGEMMA_PATH.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['gcloud', 'storage', 'rsync', '-r',
                        PALIGEMMA_GCS_URI + '/', str(PALIGEMMA_PATH) + '/'],
                       check=True)
    print('  done.')

# HF auth is only required if you want to fall back to a fresh HF download.
# With the GCS cache populated, no token is needed.
if HfFolder.get_token() is None:
    print('(no HF token cached — fine, weights load from local path)')
else:
    print('(HF token present — fallback path available)')


## 7. Define an inline PaliGemma backbone

Subclasses `Backbone` + `nn.Module`, wraps PaliGemma's vision tower (SigLIP-So400m), and exposes `extract_features` → `{'features': (B, C, h, w)}` so the existing 1×1-conv seg heads consume it unchanged.

- **Input size**: 224 (per `mapclass/configs/EVAL-03_protocol.md`)
- **Vision tower output**: `(B, num_patches, hidden_size)` reshaped to `(B, hidden_size, h, w)` where `h = w = 224 / patch_size`
- **Trainable params**: vision tower **frozen** by default — only seg heads train. Flip `freeze_vision=False` (or use the LoRA cell at the bottom) to fine-tune.

In [ ]:
import math
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor

from mapclass.model.backbone import Backbone

# Load from the GCS-cached local path (see cell above). Falls back to the
# HF model id if you want a fresh download — pass model_id='google/paligemma-3b-mix-224'.
PALIGEMMA_DEFAULT_PATH = str(PALIGEMMA_PATH)

class PaliGemmaBackbone(Backbone, nn.Module):
    """Inline Phase 5 preview — wraps PaliGemma's vision tower as a Backbone.

    Returns features at the SigLIP patch grid (16x16 for 224 input, patch_size=14).
    Seg heads upsample to the input H,W via F.interpolate.
    """
    image_size: int = 224
    processor_identity: str = 'paligemma-3b-mix-224'

    def __init__(self, model_id: str = PALIGEMMA_DEFAULT_PATH, dtype=torch.bfloat16, freeze_vision: bool = True):
        nn.Module.__init__(self)
        self.processor = PaliGemmaProcessor.from_pretrained(model_id)
        full = PaliGemmaForConditionalGeneration.from_pretrained(model_id, torch_dtype=dtype)
        # We only need the vision tower; release the LLM to free VRAM.
        self.vision_tower = full.vision_tower
        del full
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        hidden = self.vision_tower.config.hidden_size
        patch = self.vision_tower.config.patch_size
        self.feature_channels = {'features': hidden}
        self.grid = self.image_size // patch        # 16 for 224/14
        self.dtype = dtype
        if freeze_vision:
            for p in self.vision_tower.parameters():
                p.requires_grad_(False)

    def preprocess(self, image):
        """Accept PIL.Image or float tensor in [0, 1]. Returns (B, 3, 224, 224)."""
        if isinstance(image, torch.Tensor):
            # MapClassDataset emits (B, 3, H, W) floats in [0, 1].
            t = image if image.dim() == 4 else image.unsqueeze(0)
            t = torch.nn.functional.interpolate(t, size=(self.image_size, self.image_size), mode='bilinear', align_corners=False)
            # SigLIP normalization: mean=0.5, std=0.5 (same as the processor's default)
            return ((t - 0.5) / 0.5).to(dtype=self.dtype)
        # PIL.Image path — use the processor's image_processor for fidelity
        out = self.processor.image_processor(images=image, return_tensors='pt')['pixel_values']
        return out.to(dtype=self.dtype)

    def extract_features(self, image):
        # image: (B, 3, 224, 224) in self.dtype
        out = self.vision_tower(pixel_values=image)
        feats = out.last_hidden_state              # (B, num_patches, hidden)
        B, N, C = feats.shape
        h = w = int(math.isqrt(N))
        assert h * w == N, f'non-square patch grid: N={N}'
        # (B, N, C) -> (B, C, h, w); cast to float32 for the seg-head convs.
        feats = feats.transpose(1, 2).reshape(B, C, h, w).float()
        return {'features': feats}

# Construct it (loads from the GCS-cached local path; ~6 GB into VRAM in bf16).
print(f'loading PaliGemma vision tower from {PALIGEMMA_DEFAULT_PATH} ...')
pg_bb = PaliGemmaBackbone(freeze_vision=True).to(DEVICE)
print(f'  hidden_size = {pg_bb.feature_channels["features"]}, grid = {pg_bb.grid}x{pg_bb.grid}, dtype = {pg_bb.dtype}')
print(f'  vision tower params (frozen): {sum(p.numel() for p in pg_bb.vision_tower.parameters()):,}')


## 8. Train the seg heads on top of PaliGemma

Vision tower is frozen, so this is fast: only the two 1×1 convs train. Same loss as the Mock path.

In [ ]:
pg_n_feat = pg_bb.feature_channels['features']
pg_model = GeoViLM(pg_bb, LandCoverHead(in_channels=pg_n_feat), TopographyHead(in_channels=pg_n_feat)).to(DEVICE)
trainable = [p for p in pg_model.parameters() if p.requires_grad]
print(f'trainable params: {sum(p.numel() for p in trainable):,}  (vision tower frozen)')
pg_optim = torch.optim.Adam(trainable, lr=1e-3)

PG_N_EPOCHS = 2
PG_BATCH = 4   # smaller batch — PaliGemma forward is the bottleneck even when frozen
pg_loader = DataLoader(train_ds, batch_size=PG_BATCH, shuffle=True, num_workers=0)

pg_model.train()
for epoch in range(1, PG_N_EPOCHS + 1):
    losses = []
    for step, batch in enumerate(pg_loader):
        image = pg_model.backbone.preprocess(batch['image'].to(DEVICE))
        out = pg_model(image)
        loss = compute_loss(out, batch)
        pg_optim.zero_grad()
        loss.backward()
        pg_optim.step()
        losses.append(loss.item())
    print(f'PG epoch {epoch}/{PG_N_EPOCHS}: mean_loss={sum(losses)/len(losses):.4f}  (first={losses[0]:.4f}  last={losses[-1]:.4f})')

## 9. In-process PaliGemma inference

`mapclass.infer.load_model` refuses backbone != 'mock', so we run the model directly. Same softmax + WATER_TOPO overlay logic as the public predictor.

In [ ]:
from mapclass.data.taxonomy import LANDCOVER_IDX, WATER_TOPO

pg_model.eval()
with torch.no_grad():
    img = Image.open(img_path).convert('RGB')
    image = pg_model.backbone.preprocess(img).to(DEVICE)
    H, W = img.height, img.width
    # Re-upsample to the original image size so we can compare with GT masks.
    out = pg_model.backbone.extract_features(image)
    lc_logits = pg_model.lc_head(out, out_size=(H, W))[0]
    topo_logits = pg_model.topo_head(out, out_size=(H, W))[0]
    lc_probs = torch.softmax(lc_logits, dim=0)
    topo_probs = torch.softmax(topo_logits, dim=0)
    lc_argmax = lc_probs.argmax(dim=0)
    topo_argmax = topo_probs.argmax(dim=0).to(torch.uint8)
    topo_argmax[lc_argmax == LANDCOVER_IDX['water']] = WATER_TOPO

print(f'land_cover unique args: {lc_argmax.unique().tolist()}')
print(f'topography unique args (w/ water overlay 255): {torch.unique(topo_argmax).tolist()}')
print(f'land_cover prob sum at center: {lc_probs[:, H//2, W//2].sum().item():.6f}  (should be ~1.0)')

## 10. (Optional) LoRA-fine-tune PaliGemma's vision tower

If you want the vision tower to actually adapt to the maps domain (Phase 5 MODEL-04's real goal), drop LoRA adapters onto its attention layers. This is the closest the playground gets to the Phase 5 plan (`NF4 + LoRA`, per `STACK.md`).

Skip this cell to stay with the frozen-tower setup above.

In [ ]:
from peft import LoraConfig, get_peft_model

# Unfreeze, wrap with LoRA on the attention projections (SigLIP naming).
for p in pg_bb.vision_tower.parameters():
    p.requires_grad_(False)   # base stays frozen; only LoRA adapters train

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
    lora_dropout=0.05,
    bias='none',
)
pg_bb.vision_tower = get_peft_model(pg_bb.vision_tower, lora_cfg)
pg_bb.vision_tower.print_trainable_parameters()

# New optimizer covering LoRA adapters + seg heads.
trainable = [p for p in pg_model.parameters() if p.requires_grad]
print(f'total trainable (LoRA + heads): {sum(p.numel() for p in trainable):,}')
pg_optim = torch.optim.Adam(trainable, lr=5e-4)

pg_model.train()
for epoch in range(1, 3):
    losses = []
    for batch in pg_loader:
        image = pg_model.backbone.preprocess(batch['image'].to(DEVICE))
        out = pg_model(image)
        loss = compute_loss(out, batch)
        pg_optim.zero_grad()
        loss.backward()
        pg_optim.step()
        losses.append(loss.item())
    print(f'LoRA epoch {epoch}: mean_loss={sum(losses)/len(losses):.4f}')

## Where to go next

- The Mock checkpoint at `models/geovilm_phase1_mock.pt` is now available — open `playground.ipynb` and the load step will succeed.
- For PaliGemma to land as a *real* project backbone (not just an inline playground), `mapclass/model/` needs a `paligemma_backbone.py` and `train.py` / `infer.py` need the dispatch table updated (currently both hard-code `backbone='mock'`). See `mapclass/configs/EVAL-03_protocol.md` for the Phase 5 fairness constraints.
- The fixture data is synthetic noise — even a perfectly trained PaliGemma on this dataset is meaningless as a *score*. Real evaluation needs the historical / OSM splits described in `.planning/PROJECT.md`.